# GaAs calibration: which parameters give Ga:As = 1:1?

Two GaAs wafers (`F-GaAs-2`, `GaAs-SI`, proposal a321226, 2026-11-12) are
quantified with the same pipeline as `fluo_analysis_clean.ipynb` (white beam,
detector efficiency, matrix with secondary fluorescence, escape and pile-up).
A stoichiometric wafer must give `n_Ga / n_As = 1`. This notebook measures how
far the result is from 1 and which inputs could close the gap.

**Why GaAs is a demanding test.** Ga K-alpha (9.25 keV) and As K-alpha
(10.54 keV) are 1.3 keV apart, so the detection efficiency barely matters.
The matrix does:

* the Ga K edge (10.37 keV) lies between the two lines, so As K-alpha and
  K-beta are strongly absorbed by Ga on the way out, and they excite Ga
  (secondary fluorescence, ~30 % of the Ga signal);
* Ga is excited by incident photons above 10.37 keV, As only above
  11.87 keV, so the ratio depends on the **shape of the incident spectrum**
  between and above the two edges;
* Ga K-beta (10.26 keV) overlaps As K-alpha (10.54 keV).

Strategy: fit each wafer once with the measured geometry, then re-quantify the
same fitted areas while changing one input at a time (detector, geometry,
beam), and find the value of each input that gives 1:1. The K-beta/K-alpha
ratios of Ga and As give an independent check of the absorption geometry.

## 1. Settings

In [ ]:
%matplotlib inline
import dataclasses
import json
import logging
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xraylib
from scipy.optimize import brentq

from bliss_hdf5_reader import BlissMeshScan
from xrf_beamline import bm32_from_h5, per_unit_energy
from xrf_fit import (Calibration, Detector, Line, LineGroup, element_line_groups,
                     normalise_spectra, average_spectrum, counting_error,
                     fit_average_spectrum, pileup_spectra, average_pileup_template, plot_fit)
from xrf_matrix import Geometry, LabGeometry, Layer, MatrixModel, quantify_layer

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")

RAW = Path("/data/visitor/a321226/bm32/20261112/RAW_DATA")
SAMPLES = {
    "F-GaAs-2": RAW / "F-GaAs-2/F-GaAs-2_map_10x10um/F-GaAs-2_map_10x10um.h5",
    "GaAs-SI":  RAW / "GaAs-SI/GaAs-SI_map_10x10um/GaAs-SI_map_10x10um.h5",
}
SCAN_NUMBER = 1
MCA, MONITOR = "xia_det0", "mon"
OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

ELEMENTS, FAMILIES = ["Ga", "As"], ("K",)
LAYER = Layer("GaAs", 5.32, None, "wafer")     # bulk: the density cancels
REFERENCE, ANALYTE = "As K", "Ga K"
ROI_EV = (2900.0, 12600.0)
PILEUP_ROI_EV = (2900.0, 20300.0)
BKG_ORDER = 3
MIN_REL = 0.005
E_GRID_KEV = np.arange(3.0, 40.0 + 1e-9, 0.010)

# ---- measured detector geometry (2026-11)
DETECTOR_AIR_CM = 19.275         # sample-detector air path, measured
DETECTOR_AZIMUTH_DEG = 123.0     # 2theta from the GaAs 1:1 condition (take-off ~20.5 deg)
SURFACE_ANGLE_DEG = 40.0         # beam to nominal surface (incidence)
AIR_TEMP_C, AIR_PRESS_HPA = 21.0, 1013.25

DETECTOR = Detector.from_fwhm(fwhm_ev=135.0, at_energy_ev=5900.0, material="Si", fano=0.12,
                              gamma_ev=5.0, sensor_thickness_um=450.0,
                              window_material="C", window_thickness_um=1.0,
                              air_path_cm=DETECTOR_AIR_CM, air_temperature_c=AIR_TEMP_C,
                              air_pressure_hpa=AIR_PRESS_HPA)
SETUP = LabGeometry(surface_angle_deg=SURFACE_ANGLE_DEG, detector_azimuth_deg=DETECTOR_AZIMUTH_DEG)
GEOMETRY = SETUP.nominal()

# ---- incident beam (as in fluo_analysis_clean.ipynb)
BEAM_KW = dict(air_path_m=2.667, n2_path_m=0.633, temperature_c=AIR_TEMP_C, pressure_hpa=AIR_PRESS_HPA,
               kbv_angle_mrad=3.0, kbh_angle_mrad=3.0, vertical_acceptance="auto")

# ---- energy calibration: starting point, refitted on each wafer
_c = json.loads((OUT_DIR / "XRF_calib_C10_surface_scan9_energy_calibration.json").read_text())
CALIB_REF = Calibration(gain=_c["gain_ev_per_channel"], offset=_c["offset_ev"])

print(f"geometry: 2theta {SETUP.scattering_angle_deg:.1f} deg -> incidence "
      f"{GEOMETRY.incidence_deg:.2f} deg, take-off {GEOMETRY.takeoff_deg:.2f} deg")
print(f"detector: {DETECTOR_AIR_CM} cm air, {DETECTOR.sensor_thickness_um:.0f} um Si; "
      f"efficiency Ga Ka {DETECTOR.efficiency(9251.7):.4f}, As Ka {DETECTOR.efficiency(10543.7):.4f}")

## 2. Fit and quantify both wafers

For each wafer: average spectrum over the map, internal energy calibration,
pile-up resolving time, constrained fit, and self-consistent quantification
(one refit at the converged composition), exactly as in
`fluo_analysis_clean.ipynb`.

In [ ]:
def beam_spectrum(h5, **kw):
    # incident spectrum at the sample (per 0.1 % BW) from the scan's slits and mirrors
    return bm32_from_h5(h5, scan=f"{SCAN_NUMBER}.1", **{**BEAM_KW, **kw}).spectrum(E_GRID_KEV)[1]


def matrix(phi, geometry=None, secondary=True, layer=LAYER):
    return MatrixModel(layer, geometry or GEOMETRY, E_GRID_KEV, phi, secondary=secondary, target=0)


def build_groups(mm, detector=DETECTOR, roi_ev=ROI_EV):
    return element_line_groups(ELEMENTS, *roi_ev, families=FAMILIES, min_rel=MIN_REL,
                               detector=detector, matrix=mm)


def load(name):
    h5 = SAMPLES[name]
    scan = BlissMeshScan(h5_filepath=h5, scan_number=SCAN_NUMBER, verbose=False)
    scan.read_data(extra_counters=[MCA, MONITOR, f"{MCA}_live_time", f"{MCA}_elapsed_time"])
    spectra = scan.counters[MCA].astype(float)
    corrected, scale = normalise_spectra(spectra, monitor=scan.counters[MONITOR],
                                         live_time=scan.counters[f"{MCA}_live_time"])
    avg, sem, n = average_spectrum(corrected)
    return dict(name=name, h5=h5, scan=scan, spectra=spectra, scale=scale, avg=avg, n=n,
                err=counting_error(spectra, scale), nch=spectra.shape[1], phi=beam_spectrum(h5))


def analyse(d, detector=DETECTOR, geometry=None, roi_ev=ROI_EV, bkg=BKG_ORDER, tails=False,
            n_refit=1):
    # internal calibration, pile-up tau, fit, quantification
    avg, err, nch = d["avg"], d["err"], d["nch"]
    mm0 = matrix(d["phi"], geometry)
    kw = dict(yerr=err, background_order=bkg, escape=True, fit_tails=tails)
    g = build_groups(mm0, detector, roi_ev)
    roi = CALIB_REF.channel_range(*roi_ev, nch)
    cal = fit_average_spectrum(avg, g, CALIB_REF, detector, roi, **kw, fit_calibration=True).calibration
    roi = cal.channel_range(*roi_ev, nch)
    pp = average_pileup_template(
        pileup_spectra(d["spectra"], cal, d["scan"].counters[f"{MCA}_elapsed_time"]), d["scale"])
    r_pp = fit_average_spectrum(avg, build_groups(mm0, detector, PILEUP_ROI_EV), cal, detector,
                                cal.channel_range(*PILEUP_ROI_EV, nch),
                                **{**kw, "background_order": bkg + 2},
                                pileup_template=pp, fit_pileup=True)
    tau = float(r_pp.model.pileup_tau_us(r_pp.popt))
    fkw = dict(**kw, pileup_template=pp, fit_pileup=False, pileup_tau_us=tau)
    mm = mm0
    for _ in range(n_refit + 1):
        groups = {x.label: x for x in build_groups(mm, detector, roi_ev)}
        result = fit_average_spectrum(avg, list(groups.values()), cal, detector, roi, **fkw)
        quant = quantify_layer(result.areas, groups, mm0, detector, REFERENCE, [ANALYTE])
        mm = quant["matrix"]
    return dict(calibration=cal, tau_us=tau, result=result, groups=groups, quant=quant,
                ratio=quant["ratios"]["Ga"], fit_kw=fkw, roi=roi)


DATA, FITS = {}, {}
for name in SAMPLES:
    t0 = time.time()
    DATA[name] = load(name)
    FITS[name] = analyse(DATA[name])
    f = FITS[name]
    print(f"{name:<9} {DATA[name]['n']} pixels  gain {f['calibration'].gain:.5f} offset "
          f"{f['calibration'].offset:+.2f} eV  tau {f['tau_us']:.3f} us  chi2_red "
          f"{f['result'].chi2_red:.1f}  ->  n_Ga/n_As = {f['ratio']:.4f}   "
          f"(SF {100*f['quant']['enhancement']['Ga K']:.1f} % of Ga primary)  [{time.time()-t0:.0f} s]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, (name, f) in zip(axes, FITS.items()):
    r = f["result"]
    e = f["calibration"].energy(r.channels) / 1000
    ax.semilogy(e, r.y, ".", ms=2.5, color="0.35", label="data (map average)")
    ax.semilogy(e, r.yfit, "-", lw=1.5, color="#2a78d6", label="constrained fit")
    ax.set_xlim(8.5, 12.6)
    ax.set_ylim(np.percentile(r.y, 5) / 3, r.y.max() * 2)
    ax.set_title(f"{name}: chi2_red {r.chi2_red:.0f}", loc="left", fontsize=10)
    ax.set_xlabel("Energy (keV)")
    for lab, x in (("Ga Ka", 9.25), ("Ga Kb", 10.26), ("As Ka", 10.54), ("As Kb", 11.73)):
        ax.annotate(lab, (x, r.y.max() * 1.3), ha="center", fontsize=8, color="0.3")
    ax.grid(alpha=0.25, lw=0.5)
axes[0].set_ylabel("counts / live s (norm.)")
axes[0].legend(fontsize=8, loc="lower left")
fig.tight_layout()

## 3. Is the fit the problem?

The same spectra fitted with other background orders, a Hypermet tail, other
energy ranges, and with K-alpha or K-beta alone as the measure of each element
(separate free amplitudes for the K-alpha and K-beta lines of each element).

In [ ]:
def split_ka_kb(groups):
    # each K group -> a K-alpha (KL*) group and a K-beta (KM*, KN*) group
    out = []
    for g in groups:
        for fam, is_a in (("Ka", True), ("Kb", False)):
            lines = [l for l in g.lines if l.name.startswith("KL") == is_a]
            if lines:
                top = max(l.rel for l in lines)
                lines = sorted((Line(l.name, l.energy_ev, l.rel / top) for l in lines),
                               key=lambda l: -l.rel)
                out.append(LineGroup(g.element, fam, lines, g.ref_strength * top))
    return {x.label: x for x in out}


SPLIT = {}
for name, f in FITS.items():
    gs = split_ka_kb(f["groups"].values())
    rs = fit_average_spectrum(DATA[name]["avg"], list(gs.values()), f["calibration"], DETECTOR,
                              f["roi"], **f["fit_kw"])
    SPLIT[name] = (rs, gs)

variants = [("as configured", {}), ("background order 2", dict(bkg=2)),
            ("background order 4", dict(bkg=4)), ("Hypermet tail fitted", dict(tails=True)),
            ("ROI 7.5-12.6 keV", dict(roi_ev=(7500.0, 12600.0))),
            ("ROI 2.9-13.5 keV", dict(roi_ev=(2900.0, 13500.0)))]
print(f"{'fit':<24}" + "".join(f"{n:>12}" for n in SAMPLES) + "   (n_Ga/n_As)")
for lab, kw in variants:
    vals = [FITS[n]["ratio"] if not kw else analyse(DATA[n], **kw)["ratio"] for n in SAMPLES]
    print(f"{lab:<24}" + "".join(f"{v:>12.4f}" for v in vals))
for ref, an in (("As Ka", "Ga Ka"), ("As Kb", "Ga Kb")):
    vals = []
    for n in SAMPLES:
        rs, gs = SPLIT[n]
        vals.append(quantify_layer(rs.areas, gs, FITS[n]["quant"]["matrix"], DETECTOR, ref, [an])["ratios"]["Ga"])
    print(f"{an[-2:] + ' lines only':<24}" + "".join(f"{v:>12.4f}" for v in vals))

The fit choices move the ratio by less than 0.5 %. K-alpha alone and K-beta
alone differ by ~3 % (1.002 and 0.974 for F-GaAs-2), because the measured As
K-beta/K-alpha ratio is ~4 % above the model (section 5): measured on K-beta
alone, As comes out too high.

## 4. Detector and geometry parameters

Same fitted areas, re-quantified with one input changed at a time. For bulk
GaAs the K line ratios inside each group hardly depend on these inputs, so
re-quantifying without refitting is accurate here (checked in section 3: the
fit is stable).

In [ ]:
def requant(name, detector=DETECTOR, geometry=None, phi=None, secondary=True):
    f = FITS[name]
    mm = matrix(DATA[name]["phi"] if phi is None else phi, geometry, secondary)
    return quantify_layer(f["result"].areas, f["groups"], mm, detector, REFERENCE,
                          [ANALYTE])["ratios"]["Ga"]


def solve_one(fun, lo, hi):
    # parameter value giving n_Ga/n_As = 1, or None if not bracketed
    try:
        return brentq(lambda x: fun(x) - 1.0, lo, hi, xtol=1e-3)
    except ValueError:
        return None


def setup_at(two_theta):
    return dataclasses.replace(SETUP, detector_azimuth_deg=two_theta)


def angle_from_above(setup):
    # angle, seen from above, between the detector axis and the horizontal line of the
    # sample surface (what a protractor laid on the table measures)
    d = setup.detector.copy()
    d[2] = 0.0
    d /= np.linalg.norm(d)
    return float(np.degrees(np.arccos(abs(d @ setup.stage_axes()[0]))))


PARAMS = {   # label: (unit, nominal, scan values, function(name, value) -> ratio)
    "detector 2theta": ("deg", DETECTOR_AZIMUTH_DEG, np.linspace(100, 175, 11),
                        lambda n, v: requant(n, geometry=setup_at(v).nominal())),
    "take-off angle": ("deg", GEOMETRY.takeoff_deg, np.linspace(10, 60, 11),
                       lambda n, v: requant(n, geometry=Geometry(GEOMETRY.incidence_deg, v))),
    "incidence angle": ("deg", GEOMETRY.incidence_deg, np.linspace(20, 70, 11),
                        lambda n, v: requant(n, geometry=Geometry(v, GEOMETRY.takeoff_deg))),
    "detector air path": ("cm", DETECTOR_AIR_CM, np.linspace(0, 40, 9),
                          lambda n, v: requant(n, detector=dataclasses.replace(DETECTOR, air_path_cm=v))),
    "Si sensor thickness": ("um", DETECTOR.sensor_thickness_um, np.linspace(150, 1000, 11),
                            lambda n, v: requant(n, detector=dataclasses.replace(DETECTOR, sensor_thickness_um=v))),
}
BOUNDS = {"detector 2theta": (92, 179), "take-off angle": (3, 89), "incidence angle": (3, 89),
          "detector air path": (0, 100), "Si sensor thickness": (30, 3000)}

t0 = time.time()
SCANS = {p: {n: np.array([fun(n, v) for v in vals]) for n in SAMPLES}
         for p, (_, _, vals, fun) in PARAMS.items()}
SOLUTIONS = {p: {n: solve_one(lambda v: PARAMS[p][3](n, v), *BOUNDS[p]) for n in SAMPLES}
             for p in PARAMS}
print(f"({time.time()-t0:.0f} s)\n")

print(f"{'parameter':<22}{'nominal':>10}" + "".join(f"{'1:1 ' + n:>16}" for n in SAMPLES))
for p, (unit, nom, _, _) in PARAMS.items():
    sol = ["  none" if SOLUTIONS[p][n] is None else f"{SOLUTIONS[p][n]:.1f}" for n in SAMPLES]
    print(f"{p + ' (' + unit + ')':<22}{nom:>10.2f}" + "".join(f"{s:>16}" for s in sol))
print(f"\nno secondary fluorescence: " + ", ".join(f"{n} {requant(n, secondary=False):.4f}" for n in SAMPLES))

# The detector is horizontal and the surface is tilted about a horizontal axis, so the
# take-off angle is NOT the angle between the detector and the surface seen from above.
print(f"\n{'2theta':>7}{'take-off':>10}{'seen from above':>17}" + "".join(f"{n:>12}" for n in SAMPLES))
for tt in (115.0, 120.0, 122.0, 126.0, 130.0, 140.0, DETECTOR_AZIMUTH_DEG):
    st = setup_at(tt)
    print(f"{tt:>7.1f}{st.nominal().takeoff_deg:>10.2f}{angle_from_above(st):>17.2f}"
          + "".join(f"{requant(n, geometry=st.nominal()):>12.4f}" for n in SAMPLES))
for n in SAMPLES:
    st = setup_at(SOLUTIONS["detector 2theta"][n])
    print(f"{n}: 1:1 at 2theta {st.detector_azimuth_deg:.1f} deg = take-off {st.nominal().takeoff_deg:.1f} deg "
          f"= {angle_from_above(st):.1f} deg from the surface seen from above")

In [ ]:
COLORS = {"F-GaAs-2": "#2a78d6", "GaAs-SI": "#eb6834"}
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8), sharey=True)
for ax, (p, (unit, nom, vals, _)) in zip(axes, PARAMS.items()):
    for n in SAMPLES:
        ax.plot(vals, SCANS[p][n], "-o", lw=2, ms=4, color=COLORS[n], label=n)
    ax.axhline(1.0, color="0.3", lw=1, ls="--")
    ax.axvline(nom, color="0.6", lw=1, ls=":")
    ax.text(nom, 1.27, " current setting", fontsize=8, color="0.4")
    ax.set_xlabel(f"{p} ({unit})")
    ax.grid(alpha=0.25, lw=0.5)
axes[0].set_ylabel(r"$n_{\rm Ga}/n_{\rm As}$")
axes[0].set_ylim(0.85, 1.3)
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle("Ga:As vs one detector/geometry input at a time (others at their current settings); "
             "dashed: stoichiometry", fontsize=10, x=0.01, ha="left")
fig.tight_layout()
fig.savefig(OUT_DIR / "GaAs_calibration_detector_scan.png", dpi=150, bbox_inches="tight")

Reading the tables:

* With 2theta = 123 deg (take-off 20.5 deg) both wafers are within 0.3 % of
  1:1. The value of each input that gives exactly 1:1 brackets the settings:
  2theta 122.8-123.6 deg, detector air path 18.9-20.4 cm (19.275 cm
  measured), sensor 446-462 um (450 um nominal).
* The **take-off angle** is the sensitive input: ~0.4 % per degree of 2theta
  near 123 deg (table above). The detector air path moves the ratio by only
  ~0.25 % per cm.
* The detector is horizontal but the surface is tilted by 40 deg about a
  horizontal axis, so the take-off angle is **not** the angle between the
  detector and the surface seen from above: `sin(take-off) = sin 40 |cos 2theta|`,
  while the angle seen from above is `2theta - 90`. At 2theta = 123 deg the
  detector is 33 deg from the surface seen from above, but the take-off is
  20.5 deg.

## 5. Internal check: K-beta / K-alpha

The K-beta/K-alpha ratio of each element is set by the tabulated emission
rates, the detection efficiency and the absorption in GaAs on the way out. It
does not depend on the stoichiometry, so it tests the geometry independently.
As K-alpha and K-beta both lie above the Ga K edge, so the As ratio is the more
sensitive one.

In [ ]:
def line_sensitivity(mm, element, line, detector=DETECTOR):
    z = xraylib.SymbolToAtomicNumber(element)
    macro = getattr(xraylib, line + "_LINE")
    e = xraylib.LineEnergy(z, macro)
    return mm.sensitivity(z, macro, e) * detector.efficiency(1000 * e)

TAKEOFFS = np.linspace(5, 85, 17)
KBKA, KBKA_TAKEOFF = {}, {}
print(f"{'':<10}{'element':<9}{'measured KM3/KL3':>20}{f'model ({GEOMETRY.takeoff_deg:.0f} deg)':>16}"
      f"{'meas/model':>12}{'take-off fit':>14}")
for n in SAMPLES:
    rs, gs = SPLIT[n]
    mm_c = FITS[n]["quant"]["matrix"]
    for el in ELEMENTS:
        a, b = rs.areas[f"{el} Ka"], rs.areas[f"{el} Kb"]
        meas = b / a
        dmeas = meas * np.hypot(rs.area_errors[f"{el} Kb"] / b, rs.area_errors[f"{el} Ka"] / a)
        model = gs[f"{el} Kb"].ref_strength / gs[f"{el} Ka"].ref_strength
        curve = np.array([line_sensitivity(MatrixModel(mm_c.layer, Geometry(GEOMETRY.incidence_deg, t),
                                                       E_GRID_KEV, DATA[n]["phi"], target=0), el, "KM3")
                          / line_sensitivity(MatrixModel(mm_c.layer, Geometry(GEOMETRY.incidence_deg, t),
                                                         E_GRID_KEV, DATA[n]["phi"], target=0), el, "KL3")
                          for t in TAKEOFFS])
        KBKA[(n, el)] = (meas, dmeas, model, curve)
        t_match = np.interp(meas, curve[::-1], TAKEOFFS[::-1], left=np.nan, right=np.nan)
        KBKA_TAKEOFF[(n, el)] = float(t_match)
        print(f"{n:<10}{el:<9}{f'{meas:.4f} +/- {dmeas:.4f}':>20}{model:>16.4f}{meas/model:>12.3f}"
              f"{t_match:>14.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, el in zip(axes, ELEMENTS):
    for n in SAMPLES:
        meas, dmeas, model, curve = KBKA[(n, el)]
        ax.plot(TAKEOFFS, curve, "-", lw=2, color=COLORS[n], label=f"model, {n}")
        ax.axhline(meas, color=COLORS[n], lw=1, ls="--", label=f"measured, {n}")
    ax.axvline(GEOMETRY.takeoff_deg, color="0.6", lw=1, ls=":")
    ax.set_title(f"{el} K-beta1,3 / K-alpha1 (KM3/KL3)", loc="left", fontsize=10)
    ax.set_xlabel("take-off angle (deg)")
    ax.grid(alpha=0.25, lw=0.5)
axes[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUT_DIR / "GaAs_calibration_kbka.png", dpi=150, bbox_inches="tight")

At the configured take-off (20.5 deg):

* **Ga** K-beta/K-alpha agrees with the model to within 0.6 %; it would match
  exactly at a take-off of 18.7-19.7 deg. This confirms the geometry from a
  measurement that does not involve the stoichiometry.
* **As** K-beta/K-alpha is ~4 % above the model and would need a take-off of
  ~8 deg. Both As lines sit above the Ga K edge, where the absorption of GaAs
  is steepest, so the As ratio is the most sensitive to the tabulated
  absorption and emission rates; a few per cent error in the tabulated As
  K-beta/K-alpha (typical for Z ~ 30-35) shifts the matching angle a lot.
  This is also why K-beta alone gives a low Ga:As (section 3).

## 6. The incident spectrum

Photons between the Ga K edge (10.37 keV) and the As K edge (11.87 keV) excite
Ga only. The modelled spectrum in that window is shaped by the incident air
path (2.667 m of air and 0.633 m of dry N2: ~77 % absorbed at 11 keV) and by
the four Ir-coated mirrors,
whose reflectivity drops at the Ir L3 edge (11.22 keV), right between the two
K edges. The ratio is therefore a measure of the beam model as much as of the
detector.

In [ ]:
BEAM_VARIANTS = {"incident air path (m)": ("air_path_m", BEAM_KW["air_path_m"], np.linspace(0.0, 4.0, 9)),
                 "KB grazing angle (mrad)": (None, 3.0, np.linspace(2.0, 4.0, 5))}

def beam_ratio(n, key, v):
    kw = {"kbv_angle_mrad": v, "kbh_angle_mrad": v} if key is None else {key: v}
    return requant(n, phi=beam_spectrum(SAMPLES[n], **kw))

t0 = time.time()
BEAM_SCANS = {lab: {n: np.array([beam_ratio(n, key, v) for v in vals]) for n in SAMPLES}
              for lab, (key, _, vals) in BEAM_VARIANTS.items()}
AIR_SOLUTION = {n: solve_one(lambda v: beam_ratio(n, "air_path_m", v), 0.0, 4.0) for n in SAMPLES}
FULL_ACC = {n: requant(n, phi=beam_spectrum(SAMPLES[n], vertical_acceptance="full")) for n in SAMPLES}
print(f"({time.time()-t0:.0f} s)")
for n in SAMPLES:
    s = AIR_SOLUTION[n]
    print(f"{n:<9}: 1:1 for an incident air path of {s:.2f} m (model: {BEAM_KW['air_path_m']} m); "
          f"vertically integrated source: {FULL_ACC[n]:.4f}")

# the incident spectrum around the two K edges
d0 = DATA[next(iter(SAMPLES))]
bl = bm32_from_h5(d0["h5"], scan=f"{SCAN_NUMBER}.1", **BEAM_KW)
_, phi, transfers = bl.spectrum(E_GRID_KEV)
dens = per_unit_energy(E_GRID_KEV, phi)
edges = {"Ga K": xraylib.EdgeEnergy(31, xraylib.K_SHELL), "As K": xraylib.EdgeEnergy(33, xraylib.K_SHELL),
         "Ir L3": xraylib.EdgeEnergy(77, xraylib.L3_SHELL)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for L, ls in ((BEAM_KW["air_path_m"], "-"), (AIR_SOLUTION[d0["name"]], "--")):
    p = per_unit_energy(E_GRID_KEV, beam_spectrum(d0["h5"], air_path_m=L))
    ax1.plot(E_GRID_KEV, p / p.max(), ls, lw=2, color="#2a78d6",
             label=f"{L:.2f} m incident air")
for (lab, e), ha in zip(edges.items(), ("right", "left", "center")):
    ax1.axvline(e, color="0.5", lw=1, ls=":")
    ax1.text(e, 1.03 if ha != "center" else 0.06, f" {lab} ", ha=ha, fontsize=8, color="0.3")
ax1.axvspan(edges["Ga K"], edges["As K"], color="0.92", zorder=0)
ax1.set_xlim(5, 40); ax1.set_ylim(0, 1.1)
ax1.set_xlabel("Energy (keV)"); ax1.set_ylabel("photons / keV (normalised)")
ax1.set_title("Modelled incident spectrum; grey: excites Ga only", loc="left", fontsize=10)
ax1.legend(fontsize=8); ax1.grid(alpha=0.25, lw=0.5)
lab0, (key, nom, vals) = next(iter(BEAM_VARIANTS.items()))
for n in SAMPLES:
    ax2.plot(vals, BEAM_SCANS[lab0][n], "-o", lw=2, ms=4, color=COLORS[n], label=n)
ax2.axhline(1.0, color="0.3", lw=1, ls="--"); ax2.axvline(nom, color="0.6", lw=1, ls=":")
ax2.set_xlabel(lab0); ax2.set_ylabel(r"$n_{\rm Ga}/n_{\rm As}$")
ax2.legend(fontsize=8); ax2.grid(alpha=0.25, lw=0.5)
fig.tight_layout()
fig.savefig(OUT_DIR / "GaAs_calibration_beam.png", dpi=150, bbox_inches="tight")

print(f"\n{'KB angle (mrad)':<18}" + "".join(f"{v:>8.2f}" for v in BEAM_VARIANTS['KB grazing angle (mrad)'][2]))
for n in SAMPLES:
    print(f"{n:<18}" + "".join(f"{v:>8.4f}" for v in BEAM_SCANS['KB grazing angle (mrad)'][n]))

## 7. Summary and save

In [ ]:
summary = {
    "samples": {n: {"file": str(SAMPLES[n]), "ratio_Ga_As": FITS[n]["ratio"],
                    "chi2_red": FITS[n]["result"].chi2_red,
                    "secondary_enhancement_Ga": FITS[n]["quant"]["enhancement"]["Ga K"],
                    "kbka_measured_over_model": {el: KBKA[(n, el)][0] / KBKA[(n, el)][2] for el in ELEMENTS},
                    "kbka_matching_takeoff_deg": {el: KBKA_TAKEOFF[(n, el)] for el in ELEMENTS}}
                for n in SAMPLES},
    "settings": {"detector_air_cm": DETECTOR_AIR_CM, "detector_azimuth_deg": DETECTOR_AZIMUTH_DEG,
                 "incidence_deg": GEOMETRY.incidence_deg, "takeoff_deg": GEOMETRY.takeoff_deg,
                 "sensor_um": DETECTOR.sensor_thickness_um, "beam": BEAM_KW},
    "value_for_1to1": {**{p: SOLUTIONS[p] for p in PARAMS},
                       "incident air path (m)": AIR_SOLUTION},
}
print(f"{'':<34}" + "".join(f"{n:>12}" for n in SAMPLES))
print(f"{'n_Ga/n_As (measured settings)':<34}" + "".join(f"{FITS[n]['ratio']:>12.4f}" for n in SAMPLES))
for p, sol in summary["value_for_1to1"].items():
    print(f"{'1:1 needs ' + p:<34}" + "".join(f"{'none' if sol[n] is None else round(sol[n], 2):>12}" for n in SAMPLES))
out = OUT_DIR / "GaAs_calibration.json"
out.write_text(json.dumps(summary, indent=2, default=float))
print("\nwritten:", out)

## Conclusions

* Settings: detector air path 19.275 cm (measured), incident path 2.667 m of
  air plus 0.633 m of dry N2, detector at 2theta = 123 deg (take-off
  20.5 deg). With them the two wafers give `n_Ga/n_As` = **0.998** (F-GaAs-2)
  and **1.001** (GaAs-SI) from the map-averaged spectrum, and the same in
  `fluo_analysis_clean.ipynb` (map medians 0.996 and 1.001).
* 2theta = 123 deg is **calibrated on these wafers, not measured**: it is the
  angle that gives 1:1 (122.8 and 123.6 deg for the two wafers), so the 1:1
  is by construction. The independent check is the Ga K-beta/K-alpha ratio,
  which does not depend on the stoichiometry and matches the model at a
  take-off of 19-20 deg, within ~1 deg of the calibrated 20.5 deg.
* The first estimate of the geometry, "36 deg from the surface", is
  consistent with this if it was taken seen from above (2theta = 126 deg,
  take-off 22 deg, Ga:As = 1.01). The 157 deg used at first (take-off 36 deg)
  gave 1.09-1.10.
* Sensitivities near the solution: 2theta ~0.4 % per degree; detector air
  path ~0.25 % per cm; KB angle +/-0.5 mrad: +/-0.2 %; fit choices < 0.5 %.
  Modelling the 0.633 m N2 section of the incident path (instead of 3.3 m of
  air) lowered the ratio by 0.45 %.
* Open points: the As K-beta/K-alpha ratio is 4 % above the model
  (tabulated rates, or the absorption just above the Ga K edge), and the
  computed secondary fluorescence is 30 % of the Ga primary signal (the ratio
  would be 1.23 without it), so a 3 % error in it is a 1 % error in Ga:As.
* A direct measurement of 2theta (angle between the beam and the detector
  axis in the horizontal plane) would turn this calibration into a test.